# Day 09. Exercise 02
# Metrics

## 0. Imports

In [ ]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
path1 = "../../datasets/day-of-week-not-scaled.csv"
paht2 = "../../datasets/dayofweek.csv"

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [13]:
df = pd.read_csv(path1)
df2 = pd.read_csv(paht2)
df["dayofweek"] = df2["dayofweek"]
X = df.drop(columns="dayofweek")
y = df["dayofweek"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [14]:
def func(model, X = X_test, y = y_test):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, average="weighted")
    recall = recall_score(y, y_pred, average="weighted")
    roc_auc = roc_auc_score(y, y_proba, multi_class="ovo", average="weighted")
    print(f"accuracy is {accuracy:.5f}")
    print(f"precision is {precision:.5f}")
    print(f"recall is {recall:.5f}")
    print(f"roc_auc {roc_auc:.5f}")

In [15]:
svc = SVC(C = 10, class_weight=None, gamma='auto', kernel='rbf', probability=True, random_state=21)
svc.fit(X_train, y_train)
func(svc)

/home/mnemokae/Desktop/School21_projects/DSB11_ML_Advanced.ID_1577654-1/mnemokae/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc 0.97878


## 3. Decision tree

1. The same task for decision tree

In [16]:
tree = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)
tree.fit(X_train, y_train)
func(tree)

accuracy is 0.88757
precision is 0.89072
recall is 0.88757
roc_auc 0.93652


## 4. Random forest

1. The same task for random forest.

In [17]:
forest = RandomForestClassifier(class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100, random_state=21)
forest.fit(X_train, y_train)
func(forest)

accuracy is 0.92899
precision is 0.93035
recall is 0.92899
roc_auc 0.98727


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [18]:
best_model = forest
y_pred = best_model.predict(X_test)

result = pd.DataFrame({"true": y_test, "pred": y_pred})

In [19]:
users = []
for col in X_test.columns:
    if col.startswith("uid_"):
        users.append(col)

result["user"] = X_test[users].idxmax(axis=1)

In [20]:
labnames = []
for col in X_test.columns:
    if col.startswith("labname_"):
        labnames.append(col)

result["labname"] = X_test[labnames].idxmax(axis=1)

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [21]:
for weekday in result["true"].unique():
    class_data = result[result["true"] == weekday]
    total = len(class_data)
    errors = (class_data["true"] != class_data["pred"]).sum()
    error_rate = errors/total * 100
    print(f"{weekday}: {error_rate:.2f}%")

1: 9.09%
5: 5.56%
6: 1.41%
3: 5.00%
2: 6.67%
4: 14.29%
0: 22.22%
